# Классификация ботов — quickstart

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [2]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


In [3]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 328905 entries, 0 to 328904
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   cookie_id      328905 non-null  object        
 1   event_ts       328905 non-null  datetime64[ns]
 2   eid            328905 non-null  int64         
 3   event_name     328905 non-null  object        
 4   platform       328905 non-null  object        
 5   user_agent     328905 non-null  object        
 6   item_id        214308 non-null  float64       
 7   item_category  295985 non-null  object        
 8   item_location  305112 non-null  object        
 9   seller_type    195052 non-null  object        
 10  search_query   100402 non-null  object        
 11  search_page    100402 non-null  float64       
 12  pointer_x      108540 non-null  float64       
 13  pointer_y      108540 non-null  float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(8

с типами данных все хорошо

# Небольшой EDA

## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [4]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


**Типы событий.** В журнале представлены 10 типов действий.
Наиболее частые события - просмотр объявления и просмотр поисковой выдачи. Вместе они составляют около 67,3% событий.
Здесь я посчитал действия всех пользователей вместе, включая события за пределами заданных суточных окон. Пока по этим числам нельзя сказать, чем боты отличаются от людей. Например, 100 просмотров могли сделать как одна кука, так и 100 разных. 

**Платформы.** одно и тоже в разных в регистрх и альтернативные
названия: `WEB`-> `Web`-> `web` и др.
Перед построением признаков нужно привисти будет все к нижнему регистру, удалить возможные пробелы и объединить обозначения в 4 группы: `web`, `android`, `desktop`, `ios`. 

**Пропуски.** В полях `cookie_id`, `event_ts`, `eid`, `event_name`,
`platform` и `user_agent` пропусков не обнаружено.

В других призанков часть пропусков может иметь смысл события или платформой:
например, поисковый запрос неприменим ко многим действиям

## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [5]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print('Событий train внутри окон:', len(ev_tr))
print('Событий test внутри окон:', len(ev_te))
print('Всего событий внутри окон:', len(ev_tr) + len(ev_te))

Событий train внутри окон: 198436
Событий test внутри окон: 89690
Всего событий внутри окон: 288126


События за пределами окна не используем при построении признаков (по условию)

### проверим дубликаты 

In [6]:
# проверим дубликаты
for name, ev, meta in [('train', ev_tr, train),('test', ev_te, test)]:
    print(f'\n{name}:')
    print('Полных дубликатов событий:', ev.duplicated().sum())
    print('Куки без событий внутри окна:',(~meta['cookie_id'].isin(ev['cookie_id'])).sum())


train:
Полных дубликатов событий: 2952
Куки без событий внутри окна: 0

test:
Полных дубликатов событий: 1341
Куки без событий внутри окна: 0


In [7]:
# так как нашли дубликаты посмотим на них
duplicates = ev_tr.loc[
    ev_tr.duplicated(keep=False)
].sort_values(['cookie_id', 'event_ts', 'event_name'])

display(duplicates.head(20))

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
7320,ck_0032c08d8987d0ad,2026-04-10 18:13:05,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 14; SM-G991B) Appl...,1051179.0,NaN,moskva,pro,NaN,NaN,NaN,NaN,2026-04-10,2026-04-11
152927,ck_0032c08d8987d0ad,2026-04-10 18:13:05,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 14; SM-G991B) Appl...,1051179.0,NaN,moskva,pro,NaN,NaN,NaN,NaN,2026-04-10,2026-04-11
18110,ck_00389b209b37394b,2026-04-14 16:26:24,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 14; SM-A515F) Appl...,1026235.0,noutbuki,novosibirsk,private,NaN,NaN,NaN,NaN,2026-04-14,2026-04-15
74538,ck_00389b209b37394b,2026-04-14 16:26:24,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 14; SM-A515F) Appl...,1026235.0,noutbuki,novosibirsk,private,NaN,NaN,NaN,NaN,2026-04-14,2026-04-15
34299,ck_005802d8a4215738,2026-04-14 14:21:17,200,item_view,Android,Avito/122.0 (Android 13; SM-A515F) okhttp/4.11.0,1011868.0,telefony,bryansk,pro,NaN,NaN,NaN,NaN,2026-04-14,2026-04-15
71605,ck_005802d8a4215738,2026-04-14 14:21:17,200,item_view,Android,Avito/122.0 (Android 13; SM-A515F) okhttp/4.11.0,1011868.0,telefony,bryansk,pro,NaN,NaN,NaN,NaN,2026-04-14,2026-04-15
31857,ck_00708a91ae901fee,2026-04-11 07:09:31,220,seller_page_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1028856.0,uslugi,sankt-peterburg,private,NaN,NaN,778.0,68.0,2026-04-11,2026-04-12
177653,ck_00708a91ae901fee,2026-04-11 07:09:31,220,seller_page_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1028856.0,uslugi,sankt-peterburg,private,NaN,NaN,778.0,68.0,2026-04-11,2026-04-12
66425,ck_007f95db10b6eab6,2026-04-09 14:47:54,100,search_results_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,avtomobili,ekaterinburg,NaN,kia rio 2019,11.0,NaN,NaN,2026-04-09,2026-04-10
148384,ck_007f95db10b6eab6,2026-04-09 14:47:54,100,search_results_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,avtomobili,ekaterinburg,NaN,kia rio 2019,11.0,NaN,NaN,2026-04-09,2026-04-10


In [8]:
#удалаю дубликаты
ev_tr = ev_tr.drop_duplicates().copy()
ev_te = ev_te.drop_duplicates().copy()

Полностью совпадающие строки считаем повторными записями одного события и оставляем по одному экземпляру. Это позволяет не завышать число действий и не создавать искусственные нулевые интервалы. Без user_id события нельзя исключить разные действия с одинаковыми полями.

### теперь посмотрим от куда пропуски в данных

In [9]:
# где есть пропуски
fields = ['item_id', 'item_category', 'item_location', 'seller_type',
    'search_query', 'search_page', 'pointer_x', 'pointer_y']

In [10]:
missing_by_event = (ev_tr.groupby('event_name')[fields]
    .agg(lambda column: column.isna().mean()).mul(100).round(1))
display(missing_by_event)

,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
event_name,,,,,,,,
contact_chat_open,0.0,5.9,3.2,8.6,100.0,100.0,68.5,68.5
contact_message_sent,0.0,6.1,3.0,9.4,100.0,100.0,67.6,67.6
contact_phone_show,0.0,6.1,2.9,9.3,100.0,100.0,67.4,67.4
favorite_add,0.0,6.2,2.8,9.3,100.0,100.0,64.9,64.9
item_view,0.0,6.0,3.0,9.1,100.0,100.0,67.0,67.0
login,100.0,100.0,100.0,100.0,100.0,100.0,64.2,64.2
photo_swipe,0.0,5.8,3.2,8.8,100.0,100.0,65.8,65.8
search_results_view,100.0,6.1,3.1,100.0,0.0,0.0,66.8,66.8
seller_page_view,0.0,5.9,3.3,9.1,100.0,100.0,66.0,66.0


Пропуски частично объясняются типом события: запрос и номер страницы заполнены только у поиска, а `item_id` отсутствует у поиска и входа в аккаунт.

В событиях, где характеристики объявления встречаются, категория пропущена примерно в 6% случаев, локация - в 3%, тип продавца - в 9%. Надо будет понять почему так.

Координаты отсутствуют в 64–69% событий каждого типа. Возможно это связано с платформой.\
Еще пропало событие `captcha_shown`

In [11]:
print('Капчи в исходных событиях train:')
print(events.loc[events['cookie_id'].isin(train['cookie_id']),'event_name'].eq('captcha_shown').sum())

print('Капчи внутри окон train:')
print(ev_tr['event_name'].eq('captcha_shown').sum())

Капчи в исходных событиях train:
7928
Капчи внутри окон train:
0


Все 7928 показов капчи куки оказались вне окон. Использовать их для построения признаков нельзя

In [12]:
# для более удобного анализа привидем все названия платформ к нижнему регистру
ev_tr['platform_norm'] = (ev_tr['platform'].str.strip().str.lower())
ev_te['platform_norm'] = (ev_te['platform'].str.strip().str.lower())

In [13]:
# теперь проверим связь платформы и пропусков в кординатах
missing_by_platform = (ev_tr.groupby('platform_norm')[['pointer_x', 'pointer_y']]
    .agg(lambda column: column.isna().mean()).mul(100).round(1))

display(missing_by_platform)

,pointer_x,pointer_y
platform_norm,,
android,100.0,100.0
desktop,39.8,39.8
ios,100.0,100.0
iphone,100.0,100.0
web,39.8,39.8


На мобильных платформах координаты полностью отсутствуют. У `web` и `desktop` пропуски составляют 39,8%. Следовательно, отсутствие координат связано с платформой и само по себе не указывает на бота. Заменять пропуски нулями нельзя: ноль может быть реальной координатой.

In [14]:
# обьеденим `iphone` и `ios` в общую группу `ios` так как iphone работает на ios
for ev in [ev_tr, ev_te]: ev['platform_group'] = ev['platform_norm'].replace({'iphone': 'ios'})

### Посмотрим на User-Agent

In [15]:
print('Уникальных User-Agent в train:', ev_tr['user_agent'].nunique())

with pd.option_context('display.max_colwidth', None):
    display(
        ev_tr['user_agent']
        .value_counts()
        .head(15)
        .rename_axis('user_agent')
        .reset_index(name='n_events')
    )

Уникальных User-Agent в train: 148


,user_agent,n_events
0,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36",2284
1,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36",2222
2,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",2212
3,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 YaBrowser/23.9.0.0 Safari/537.36",2149
4,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 YaBrowser/23.3.0.0 Safari/537.36",2144
5,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",2135
6,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 YaBrowser/23.8.0.0 Safari/537.36",2130
7,"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/118.0.0.0 Safari/537.36",2080
8,"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/114.0.0.0 Safari/537.36",2064
9,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",2047


В обучающих событиях встречаются 148 уникальных User-Agent. Среди наиболее частых - строки Яндекс Браузера, Chrome, Firefox
и HeadlessChrome. Часть строк отличается только версиями. Для признаков рассмотрим семейство клиента и операционную систему,
чтобы не полагаться только на точное совпадение полной строки. HeadlessChrome может указывать на автоматизацию, но сам по себе не подтверждает целевой класс.

In [16]:
for platform in ['desktop', 'web']:
    print(f'\nПлатформа: {platform}')

    with pd.option_context('display.max_colwidth', None):
        display(
            ev_tr.loc[
                ev_tr['platform_norm'].eq(platform),
                'user_agent'
            ]
            .value_counts()
            .head(5)
            .rename_axis('user_agent')
            .reset_index(name='n_events')
        )


Платформа: desktop


,user_agent,n_events
0,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36",611
1,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 YaBrowser/23.3.0.0 Safari/537.36",569
2,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",569
3,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36",567
4,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",558



Платформа: web


,user_agent,n_events
0,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36",1673
1,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36",1655
2,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",1643
3,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",1626
4,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 YaBrowser/23.8.0.0 Safari/537.36",1597


Среди наиболее частых User-Agent у `desktop` и `web` встречаются одинаковые браузерные строки. Доля пропусков координат у этих групп также совпадает после округления. Поэтому обьеденим эти группы.

In [17]:
for ev in [ev_tr, ev_te]: ev['platform_group'] = ev['platform_norm'].replace({'desktop': 'web','iphone': 'ios'})

### Посмотрим описательную статистику событий

In [18]:
ev_tr.describe()

,event_ts,eid,item_id,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
count,195484,195484.000000,1.308300e+05,60806.000000,65283.000000,65283.000000,195484,195484
mean,2026-04-12 19:57:37.309181440,195.668975,1.030001e+06,2.875308,929.889114,538.063906,2026-04-12 05:48:48.190542080,2026-04-13 05:48:48.190542336
min,2026-04-06 00:02:57,100.000000,1.000000e+06,1.000000,0.000000,0.000000,2026-04-06 00:00:00,2026-04-07 00:00:00
25%,2026-04-09 14:58:21.750000128,100.000000,1.014981e+06,1.000000,465.000000,280.000000,2026-04-09 00:00:00,2026-04-10 00:00:00
50%,2026-04-12 16:16:21.500000,200.000000,1.030024e+06,2.000000,908.000000,535.000000,2026-04-12 00:00:00,2026-04-13 00:00:00
75%,2026-04-15 22:40:59.249999872,210.000000,1.044989e+06,4.000000,1391.000000,796.000000,2026-04-15 00:00:00,2026-04-16 00:00:00
max,2026-04-19 23:59:50,500.000000,1.059999e+06,63.000000,1920.000000,1080.000000,2026-04-19 00:00:00,2026-04-20 00:00:00
std,NaN,89.311536,1.728826e+04,2.824327,544.341504,306.086444,NaN,NaN


In [19]:
ev_te.describe()

,event_ts,eid,item_id,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
count,88349,88349.000000,5.894500e+04,27673.000000,28975.000000,28975.000000,88349,88349
mean,2026-04-23 18:43:51.185899008,195.035813,1.029914e+06,2.942616,917.728283,536.483417,2026-04-23 04:39:37.534550528,2026-04-24 04:39:37.534550784
min,2026-04-20 00:02:10,100.000000,1.000002e+06,1.000000,0.000000,0.000000,2026-04-20 00:00:00,2026-04-21 00:00:00
25%,2026-04-21 21:46:05,100.000000,1.014867e+06,1.000000,457.000000,281.000000,2026-04-21 00:00:00,2026-04-22 00:00:00
50%,2026-04-23 20:53:29,200.000000,1.029938e+06,2.000000,891.000000,536.000000,2026-04-23 00:00:00,2026-04-24 00:00:00
75%,2026-04-25 14:50:03,210.000000,1.044841e+06,4.000000,1369.000000,792.000000,2026-04-25 00:00:00,2026-04-26 00:00:00
max,2026-04-26 23:59:50,500.000000,1.059999e+06,48.000000,1920.000000,1080.000000,2026-04-26 00:00:00,2026-04-27 00:00:00
std,NaN,88.914668,1.730812e+04,2.940204,540.777892,304.547058,NaN,NaN


После фильтрации по окнам и удаления дубликатов осталось 195 484 события в train и 88 349 в test.

События train 6-19 апреля, test - к 20-26 апреля. Тест расположен позже обучения, поэтому для проверки модели будем использовать временное разбиение.

Распределение страниц поиска близко: медиана - 2, 75 процентиль - 4 в обеих выборках. Максимум составляет 63 в train и 48 в test. Большие номера это врядли ошибка в данных, скорее может отражать ботов либо гулбино вовлеченности.

Заполненные координаты лежат в диапазонах 0–1920 по X и 0–1080 по Y в обеих выборках. Отрицательных значений нет. Медианы и квартили координат также близки.

По этим сводным статистикам крупных различий в страницах поиска и координатах не видно, но это не гарантирует одинакового поведения куки в train и test.

### Теперь перейдём к количеству событий на одну куку. 
До этого мы изучали отдельные события, а предсказывать нужно класс куки.

In [20]:
counts_tr = ( ev_tr.groupby('cookie_id').size().reindex(train['cookie_id'], fill_value=0))

counts_te = (ev_te.groupby('cookie_id').size().reindex(test['cookie_id'], fill_value=0))

display(pd.DataFrame({'train': counts_tr.describe(percentiles=[0.5,0.9, 0.95,0.99]),
                      'test': counts_te.describe(percentiles=[0.5, 0.9,0.95, 0.99])}))

,train,test
count,11091.000000,4909.000000
mean,17.625462,17.997352
std,17.437804,18.369881
min,1.000000,1.000000
50%,12.000000,12.000000
90%,39.000000,39.000000
95%,52.000000,55.000000
99%,86.000000,91.000000
max,165.000000,187.000000


В обеих выборках медиана активности составляет 12 событий на куку, 90 процентиль - 39. Среднее также близки.

В test несколько выше верхние процентили и максимум активности. По этим статистикам существенного изменения типичной активности не видно. Наиболее активные куки сохраняем: большое количество событий само по себе не является ошибкой.
### сравним ботов и людей на данных до 17 апреля, ставив последние три дня для валидации

In [21]:
eda_cookies = train.loc[train['window_start_ts'] < pd.Timestamp('2026-04-17')].copy()

eda_cookies['n_events'] = (eda_cookies['cookie_id'].map(counts_tr))

display(eda_cookies.groupby('target')['n_events'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

,count,mean,std,min,50%,90%,95%,99%,max
target,,,,,,,,,
0,8401.0,16.461493,15.747486,1.0,11.0,36.0,48.0,76.00,137.0
1,739.0,29.431664,27.304399,1.0,20.0,68.0,87.0,125.62,165.0


На обучающей части данных куки положительного класса обычно совершают больше действий: медиана составляет 20 событий против
11 у человеческого трафика, среднее - 29,4 против 16,5.

Однако количество событий не позволяет однозначно разделить классы: в обоих встречаются малоактивные куки, а высокая активность наблюдается и у людей. Сохраним количество событий как признаки дополнительно изучим состав действий.

In [22]:
event_counts = pd.crosstab(ev_tr['cookie_id'],ev_tr['event_name'])
event_shares = event_counts.div(event_counts.sum(axis=1), axis=0)
eda_shares = eda_cookies[['cookie_id', 'target']].merge(event_shares,left_on='cookie_id',
                                                        right_index=True,validate='one_to_one')
display(eda_shares.groupby('target')[event_shares.columns].mean().T.mul(100)
    .round(1).rename(columns={0: 'Люди, %', 1: 'Боты, %'}))

target,"Люди, %","Боты, %"
contact_chat_open,1.7,1.8
contact_message_sent,0.8,1.0
contact_phone_show,3.1,3.2
favorite_add,6.8,3.2
item_view,36.3,42.3
login,2.3,1.0
photo_swipe,12.7,7.9
search_results_view,30.9,33.3
seller_page_view,5.3,6.2


У ботов в среднем выше доля просмотров объявлений, но ниже доли добавлений в избранное, листаний фотографий  и входов в аккаунт, что в принципе типично для бота.
Сохраним доли типов событий как кандидаты в признаки. Различия средних не гарантируют хорошего разделения отдельных куки

### посмотрим на event_ts
Проверим, различаются ли темп и регулярность действий у людей и ботов действиями или различные регулярные активности. Проверим это 

In [23]:
eda_events = (ev_tr.loc[ev_tr['cookie_id'].isin(eda_cookies['cookie_id'])].sort_values(['cookie_id', 'event_ts']).copy())

eda_events['gap_seconds'] = (eda_events.groupby('cookie_id')['event_ts'].diff().dt.total_seconds())

gap_features = (eda_events.groupby('cookie_id')['gap_seconds'].agg(n_intervals='count',gap_median='median'
                                                                   ,gap_mean='mean',gap_std='std'))

gap_features['gap_cv'] = (gap_features['gap_std']/ gap_features['gap_mean'].replace(0, np.nan))

eda_gaps = eda_cookies[['cookie_id', 'target']].merge(gap_features,left_on='cookie_id',
                                                      right_index=True,validate='one_to_one')

display(eda_gaps.groupby('target')[['gap_median', 'gap_mean', 'gap_std', 'gap_cv']
    ].median().rename(index={0: 'Люди', 1: 'Боты'}))

,gap_median,gap_mean,gap_std,gap_cv
target,,,,
Люди,61.0,676.472727,1751.089300,2.042512
Боты,25.0,429.428571,1372.275723,2.632327


У ботов типичные интервалы между действиями короче: медиана индивидуальных медианных интервалов составляет 25 секунд, у людей - 61 секунду.

При этом относительный разброс интервалов у ботов выше: медиана коэффициента вариации — 2,63 против 2,04.
Следовательно, более быстрые действия не означают более равномерный ритм. Длинные перерывы могут влиять на среднее и разброс интервалов. Сохраним характеристики интервалов как кандидаты в признаки.

### Быстрые действия и длинные перерывы

In [24]:
intervals = eda_events.loc[ eda_events['gap_seconds'].notna(), ['cookie_id', 'gap_seconds']].copy()

intervals['gap_zero'] = intervals['gap_seconds'].eq(0)
intervals['gap_le_1s'] = intervals['gap_seconds'].le(1)
intervals['gap_le_5s'] = intervals['gap_seconds'].le(5)
intervals['gap_le_10s'] = intervals['gap_seconds'].le(10)
intervals['gap_gt_30m'] = intervals['gap_seconds'].gt(30 * 60)

share_cols = ['gap_zero', 'gap_le_1s', 'gap_le_5s','gap_le_10s', 'gap_gt_30m']
gap_shares = intervals.groupby('cookie_id')[share_cols].mean()

eda_gap_shares = eda_cookies[['cookie_id', 'target']].merge(gap_shares, left_on='cookie_id',
                                                            right_index=True,how='left',validate='one_to_one')

display( eda_gap_shares.groupby('target')[share_cols].mean().T.mul(100).round(1)
    .rename(columns={0: 'Люди, %', 1: 'Боты, %'}))

target,"Люди, %","Боты, %"
gap_zero,0.7,0.2
gap_le_1s,1.8,0.7
gap_le_5s,5.9,9.3
gap_le_10s,11.1,25.1
gap_gt_30m,14.3,8.6


У ботов выше средняя доля интервалов до 5 секунд (9,3% против 5,9%)
и до 10 секунд (25,1% против 11,1%). При этом нулевые интервалы и интервалы до 1 секунды чаще встречаются у людей.

Длинные перерывы более 30 минут у ботов занимают меньшую долю.Таким образом, различия зависят от выбранного диапазона интервалов.
Возможно это кандидаты в признаки, но не будем использовать отдельный порог скорости.

In [25]:
#посмотрим все ли впорядки с локациями
location_stats = (eda_events.groupby('item_location').agg(n_events=('cookie_id', 'size'),
        n_cookies=('cookie_id', 'nunique')).sort_values('n_cookies', ascending=False))
display(location_stats)

,n_events,n_cookies
item_location,,
novosibirsk,19086,4451
sankt-peterburg,20040,4413
moskva,18635,4400
ekaterinburg,18558,4377
izhevsk,2254,1150
krasnodar,2200,1140
kirov,2182,1136
barnaul,2252,1133
astrahan,2273,1121


In [26]:
diversity = (
    eda_events.groupby('cookie_id')
    .agg(n_events=('event_name', 'size'),n_items=('item_id', 'nunique'),n_categories=('item_category', 'nunique'),
        n_locations=('item_location', 'nunique'),n_queries=('search_query', 'nunique'),
         search_page_max=('search_page', 'max')))

# Доля уникальных объявлений среди просмотров объявлений
item_views = (eda_events.loc[eda_events['event_name'].eq('item_view')]
    .groupby('cookie_id').agg(n_item_views=('event_name', 'size'),n_viewed_items=('item_id', 'nunique')))

item_views['unique_item_ratio'] = (item_views['n_viewed_items'] / item_views['n_item_views'])

diversity = diversity.join(item_views[['unique_item_ratio']])

eda_diversity = eda_cookies[['cookie_id', 'target']].merge(
    diversity,left_on='cookie_id',right_index=True,validate='one_to_one')

display(
    eda_diversity.groupby('target')[['n_items', 'n_categories', 'n_locations','n_queries',
                                     'search_page_max', 'unique_item_ratio']]
    .median().T.round(2).rename(columns={0: 'Люди', 1: 'Боты'}))

target,Люди,Боты
n_items,6.0,11.00
n_categories,2.0,2.00
n_locations,4.0,8.00
n_queries,3.0,4.00
search_page_max,4.0,7.00
unique_item_ratio,1.0,0.88


У ботов выше медианы количества уникальных объявлений и локаций, а также глубины поиска. При этом доля уникальных объявлений среди росмотров ниже, что указывает на наличие повторных просмотров.

Количество уникальных объектов может зависеть от общей активности. Поэтому полезность этих признаков оценим совместно с количеством событий на валидации.

### Возраст куки

In [27]:
eda_cookies['cookie_age_days'] = (eda_cookies['window_end_ts']- eda_cookies['cookie_created_at']
                                 ).dt.total_seconds() / 86400

print('Куки с отрицательным возрастом:',eda_cookies['cookie_age_days'].lt(0).sum())

display(eda_cookies.groupby('target')['cookie_age_days'].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .round(2).rename(index={0: 'Люди', 1: 'Боты'}))

Куки с отрицательным возрастом: 0


,count,mean,std,min,25%,50%,75%,90%,max
target,,,,,,,,,
Люди,8401.0,75.44,67.27,1.01,20.55,63.29,111.89,165.39,440.09
Боты,739.0,51.06,63.04,1.01,2.01,22.51,86.23,142.58,362.76


медиана возраста куки у ботов составляет 22,51 дня, у людей - 63,29 дня. Нижний квартиль у ботов - 2,01 дня,
у людей - 20,55 дня. Таким образом, положительный класс чаще представлен более молодыми куки, однако диапазоны возрастов пересекаются. Добавим возраст в набор признаков.

# Простроение признаков

In [28]:
def build_features(meta, ev):
    ev = ev.sort_values(['cookie_id', 'event_ts']).copy()
    g = ev.groupby('cookie_id')

    # Сохраняем все куки и порядок из метаданных
    features = meta[['cookie_id']].set_index('cookie_id').copy()

    features['cookie_age_days'] = (
        (meta['window_end_ts'] - meta['cookie_created_at'])
        .dt.total_seconds()
        .to_numpy() / 86400
    )

    # Количество событий и разнообразие объектов
    counts = g.agg(
        n_events=('event_name', 'size'),
        n_items=('item_id', 'nunique'),
        n_categories=('item_category', 'nunique'),
        n_locations=('item_location', 'nunique'),
        n_queries=('search_query', 'nunique')
    )

    features = features.join(counts)
    features[counts.columns] = features[counts.columns].fillna(0)

    features = features.join(
        g['search_page'].max().rename('search_page_max')
    )

    # Фиксированный список типов действий из описания данных
    event_types = [
        'item_view', 'search_results_view', 'photo_swipe',
        'favorite_add', 'seller_page_view', 'contact_phone_show',
        'contact_chat_open', 'contact_message_sent', 'login'
    ]

    event_counts = pd.crosstab(
        ev['cookie_id'], ev['event_name']
    ).reindex(
        index=features.index,
        columns=event_types,
        fill_value=0
    )

    event_shares = event_counts.div(
        features['n_events'].replace(0, np.nan), axis=0
    )

    features = features.join(event_counts.add_prefix('count_'))
    features = features.join(event_shares.add_prefix('share_'))

    # Повторные просмотры объявлений
    views = (
        ev.loc[ev['event_name'].eq('item_view')]
        .groupby('cookie_id')['item_id']
        .agg(['size', 'nunique'])
    )

    features['unique_item_ratio'] = (
        views['nunique'] / views['size']
    )

    # Интервалы между соседними событиями
    ev['gap_seconds'] = g['event_ts'].diff().dt.total_seconds()

    gap_stats = ev.groupby('cookie_id')['gap_seconds'].agg(
        n_intervals='count',
        gap_median='median',
        gap_mean='mean',
        gap_std='std'
    )

    gap_stats['gap_cv'] = (
        gap_stats['gap_std']
        / gap_stats['gap_mean'].replace(0, np.nan)
    )

    features = features.join(gap_stats)
    features['n_intervals'] = features['n_intervals'].fillna(0)

    # Первые события исключаем: перед ними нет интервала
    intervals = ev.loc[
        ev['gap_seconds'].notna(),
        ['cookie_id', 'gap_seconds']
    ].copy()

    intervals['gap_zero'] = intervals['gap_seconds'].eq(0)
    intervals['gap_le_1s'] = intervals['gap_seconds'].le(1)
    intervals['gap_le_5s'] = intervals['gap_seconds'].le(5)
    intervals['gap_le_10s'] = intervals['gap_seconds'].le(10)
    intervals['gap_gt_30m'] = intervals['gap_seconds'].gt(1800)

    share_cols = [
        'gap_zero', 'gap_le_1s', 'gap_le_5s',
        'gap_le_10s', 'gap_gt_30m'
    ]

    features = features.join(
        intervals.groupby('cookie_id')[share_cols].mean()
    )

    return features

In [29]:
X_train = build_features(train, ev_tr)
X_test = build_features(test, ev_te)

assert X_train.index.equals(pd.Index(train['cookie_id']))
assert X_test.index.equals(pd.Index(test['cookie_id']))
assert X_train.columns.equals(X_test.columns)
assert X_train.index.is_unique and X_test.index.is_unique

print('Размер train:', X_train.shape)
print('Размер test:', X_test.shape)

display(X_train.head())

Размер train: (11091, 36)
Размер test: (4909, 36)


,cookie_age_days,n_events,n_items,n_categories,n_locations,n_queries,search_page_max,count_item_view,count_search_results_view,count_photo_swipe,...,n_intervals,gap_median,gap_mean,gap_std,gap_cv,gap_zero,gap_le_1s,gap_le_5s,gap_le_10s,gap_gt_30m
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,136.603692,7,4,3,5,3,6.0,3,3,1,...,6,21.0,27.833333,29.949402,1.076026,0.0,0.166667,0.166667,0.333333,0.000000
ck_7e4de46eeab82974,195.576111,41,19,1,12,5,8.0,22,8,4,...,40,57.5,986.900000,2416.286861,2.448360,0.0,0.000000,0.075000,0.200000,0.150000
ck_9320229ef6304522,33.994421,36,19,7,9,10,10.0,7,13,3,...,35,22.0,1042.600000,2306.449400,2.212209,0.0,0.028571,0.114286,0.171429,0.171429
ck_30ccd25bc1714ed9,1.546759,21,10,1,4,3,2.0,14,3,0,...,20,54.5,861.050000,2502.081501,2.905849,0.0,0.000000,0.000000,0.050000,0.100000
ck_a77c5f05948cdeef,95.837095,28,16,4,6,6,7.0,12,6,4,...,27,52.0,707.333333,1748.439172,2.471874,0.0,0.000000,0.000000,0.000000,0.111111


# Валидация

Тест лежит позже трейна по времени, поэтому и валидацию делать по времени, а не
случайным сплитом. Метрику беру из `metric.py` .

### Случайный лес

In [30]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from metric import precision_at_recall

is_valid = train.window_start_ts.ge('2026-04-17').values
y = train.target.values

model_features = Pipeline([
    ('imputer', SimpleImputer(
        strategy='median',
        add_indicator=True
    )),
    ('model', RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=0,
        n_jobs=-1
    ))
])

model_features.fit(
    X_train.loc[~is_valid],
    y[~is_valid]
)

p_valid_features = model_features.predict_proba(
    X_train.loc[is_valid]
)[:, 1]

print(
    'P@R0.7 с новыми признаками:',
    round(precision_at_recall(y[is_valid], p_valid_features), 4)
)
print(
    'Доля ботов на валидации:',
    round(y[is_valid].mean(), 4)
)

P@R0.7 с новыми признаками: 0.3972
Доля ботов на валидации: 0.082


### Градиентный бустинг

In [31]:
dates = train['window_start_ts']

fit_early = dates.lt('2026-04-14').to_numpy()
valid_early = (
    dates.ge('2026-04-14') & dates.lt('2026-04-17')
).to_numpy()

In [32]:
from sklearn.ensemble import HistGradientBoostingClassifier

splits = {
    '14–16 апреля': (fit_early, valid_early),
    '17–19 апреля': (~is_valid, is_valid)
}

boosting_results = []
boosting_models = {}

for period, (fit_mask, valid_mask) in splits.items():
    boosting = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=0
    )

    boosting.fit(
        X_train.loc[fit_mask],
        y[fit_mask]
    )

    predictions = boosting.predict_proba(
        X_train.loc[valid_mask]
    )[:, 1]

    boosting_results.append({
        'Период': period,
        'P@R≥0.7': precision_at_recall(
            y[valid_mask], predictions
        ),
        'Доля ботов': y[valid_mask].mean()
    })

    boosting_models[period] = boosting

display(
    pd.DataFrame(boosting_results)
    .set_index('Период')
    .round(4)
)

,P@R≥0.7,Доля ботов
Период,,
14–16 апреля,0.4164,0.0824
17–19 апреля,0.4828,0.0820


In [33]:
from sklearn.base import clone

rf_results = []

for period, (fit_mask, valid_mask) in splits.items():
    for name, columns in {
        'Baseline': ['n_events', 'n_items'],
        'Random Forest — расширенные признаки': X_train.columns.tolist()
    }.items():
        candidate = clone(model_features)
        candidate.fit(X_train.loc[fit_mask, columns], y[fit_mask])

        predictions = candidate.predict_proba(
            X_train.loc[valid_mask, columns]
        )[:, 1]

        rf_results.append({
            'Период': period,
            'Модель': name,
            'P@R≥0.7': precision_at_recall(y[valid_mask], predictions)
        })

display(pd.DataFrame(rf_results))

,Период,Модель,P@R≥0.7
0,14–16 апреля,Baseline,0.099639
1,14–16 апреля,Random Forest — расширенные признаки,0.365789
2,17–19 апреля,Baseline,0.095718
3,17–19 апреля,Random Forest — расширенные признаки,0.397163


Градиентный бустинг показал лучший результат на обоих временных разбиениях: 0,4164 и 0,4828 против 0,3658 и 0,3972 у Random Forest. Проверю, можно ли улучшить результат. Псомторию как влияют разные группы на качество.

In [34]:
feature_groups = {'возраст': ['cookie_age_days'],
    'интервалы': [col for col in X_train.columns
        if col.startswith('gap_') or col == 'n_intervals'],

    'состав действий': [col for col in X_train.columns
        if col.startswith(('count_', 'share_'))],

    'разнообразие и поиск': ['n_items', 'n_categories', 'n_locations',
        'n_queries', 'search_page_max', 'unique_item_ratio']
}

template = boosting_models['17–19 апреля']

full_scores = { row['Период']: row['P@R≥0.7']
    for row in boosting_results}

ablation_results = []

for period, (fit_mask, valid_mask) in splits.items():
    for group_name, excluded in feature_groups.items():
        columns = [
            col for col in X_train.columns
            if col not in excluded]

        candidate = clone(template)

        candidate.fit(X_train.loc[fit_mask, columns],y[fit_mask])
        predictions = candidate.predict_proba(X_train.loc[valid_mask, columns])[:, 1]
        score = precision_at_recall(y[valid_mask], predictions)

        ablation_results.append({'Период': period,'Убрали группу': group_name,
            'P@R≥0.7': score,'Изменение к полной модели': score - full_scores[period]})

ablation_results = pd.DataFrame(ablation_results)

display(ablation_results.set_index(['Период', 'Убрали группу']).round(4))

P@R≥0.7  Изменение к полной модели
Период       Убрали группу                                           
14–16 апреля возраст                0.4554                     0.0390
             интервалы              0.2991                    -0.1173
             состав действий        0.3605                    -0.0559
             разнообразие и поиск   0.3870                    -0.0294
17–19 апреля возраст                0.4667                    -0.0161
             интервалы              0.2743                    -0.2085
             состав действий        0.4154                    -0.0673
             разнообразие и поиск   0.3846                    -0.0981

Наибольшее падение качества произошло после удаление интервалов. Состав действий, разнообразие  также полезны
на обоих периодах. По возрасту пока не ясно. Пока оставляем полный набор.

### сравним HistGradientBoosting с CatBoost
так как качество Градиентного бустинга все мало, попробудем использовать Catboost

In [35]:
from catboost import CatBoostClassifier

catboost_results = []
catboost_models = {}

for period, (fit_mask, valid_mask) in splits.items():
    candidate = CatBoostClassifier(
        iterations=600,
        depth=5,
        learning_rate=0.05,
        l2_leaf_reg=5,
        loss_function='Logloss',
        random_seed=0,
        thread_count=1,
        verbose=False,
        allow_writing_files=False
    )

    candidate.fit(X_train.loc[fit_mask],y[fit_mask])

    predictions = candidate.predict_proba(X_train.loc[valid_mask])[:, 1]

    catboost_results.append({'Период': period,'P@R≥0.7': precision_at_recall(y[valid_mask], predictions)})
    catboost_models[period] = candidate

display(pd.DataFrame(catboost_results).set_index('Период').round(4))

,P@R≥0.7
Период,
14–16 апреля,0.4255
17–19 апреля,0.5463


CatBoost показал лучший результат на обоих временных разбиениях: 0,4255 и 0,5463. Выбираем его основным кандидатом для финального обучения. Преимущество заметнее на позднем периоде

Попробую подобрать по сетке различные гиперпараметры. Напишишу функцию для подбора, чтобы потом ее использовать и не засорять кодом ячейки

In [36]:
from sklearn.model_selection import ParameterGrid

def search_catboost(X, y, splits, param_grid, fixed_params=None):
    base_params = {
        'iterations': 600,
        'depth': 7,
        'learning_rate': 0.05,
        'l2_leaf_reg': 5,
        'loss_function': 'Logloss',
        'random_seed': 0,
        'thread_count': 1,
        'verbose': False,
        'allow_writing_files': False
    }
    base_params.update(fixed_params or {})

    combinations = list(ParameterGrid(param_grid))
    rows = []

    for step, params in enumerate(combinations, start=1):
        model_params = {**base_params, **params}
        row = {key: model_params[key]
               for key in [
                'depth', 'iterations', 'learning_rate', 'l2_leaf_reg']
        }
        scores = []

        for period, (fit_mask, valid_mask) in splits.items():
            model = CatBoostClassifier(**model_params)
            model.fit(X.loc[fit_mask], y[fit_mask])
            predictions = model.predict_proba(X.loc[valid_mask])[:, 1]
            score = precision_at_recall(y[valid_mask], predictions)
            row[period] = score
            scores.append(score)

        row['mean_score'] = float(np.mean(scores))
        row['min_score'] = float(np.min(scores))
        rows.append(row)

        print(f'[{step}/{len(combinations)}] {params}: 'f"среднее={row['mean_score']:.4f}, "
            f"минимум={row['min_score']:.4f}",flush=True)

    results = (pd.DataFrame(rows).sort_values(['mean_score', 'min_score', 'depth',
    'iterations', 'l2_leaf_reg', 'learning_rate'],ascending=[False, False, True, True, True, True]).reset_index(drop=True))
    best_params = {**base_params,'depth': int(results.loc[0, 'depth']),
        'iterations': int(results.loc[0, 'iterations']),'learning_rate': float(results.loc[0, 'learning_rate']),
        'l2_leaf_reg': float(results.loc[0, 'l2_leaf_reg'])}
    return results, best_params

In [37]:
search_results, best_behavior_params = search_catboost(
    X_train,
    y,
    splits,
    param_grid={
        'depth': [4, 5, 6, 7],
        'l2_leaf_reg': [5, 10]
    }
)
display(search_results.round(4))

[1/8] {'depth': 4, 'l2_leaf_reg': 5}: среднее=0.4794, минимум=0.4177
[2/8] {'depth': 4, 'l2_leaf_reg': 10}: среднее=0.4914, минимум=0.4228
[3/8] {'depth': 5, 'l2_leaf_reg': 5}: среднее=0.4859, минимум=0.4255
[4/8] {'depth': 5, 'l2_leaf_reg': 10}: среднее=0.4942, минимум=0.4448
[5/8] {'depth': 6, 'l2_leaf_reg': 5}: среднее=0.4966, минимум=0.4477
[6/8] {'depth': 6, 'l2_leaf_reg': 10}: среднее=0.4899, минимум=0.4613
[7/8] {'depth': 7, 'l2_leaf_reg': 5}: среднее=0.5310, минимум=0.4875
[8/8] {'depth': 7, 'l2_leaf_reg': 10}: среднее=0.5085, минимум=0.4570


,depth,iterations,learning_rate,l2_leaf_reg,14–16 апреля,17–19 апреля,mean_score,min_score
0,7,600,0.05,5,0.4875,0.5744,0.5310,0.4875
1,7,600,0.05,10,0.4570,0.5600,0.5085,0.4570
2,6,600,0.05,5,0.4477,0.5455,0.4966,0.4477
3,5,600,0.05,10,0.4448,0.5437,0.4942,0.4448
4,4,600,0.05,10,0.4228,0.5600,0.4914,0.4228
5,6,600,0.05,10,0.4613,0.5185,0.4899,0.4613
6,5,600,0.05,5,0.4255,0.5463,0.4859,0.4255
7,4,600,0.05,5,0.4177,0.5411,0.4794,0.4177


#### добавим фичи 
выделем из user_agent разные группы браузеров

In [38]:
def build_client_features(meta, ev):
    ua = ev['user_agent'].fillna('').str.lower()

    client = np.select(
        [
            ua.str.contains('avito/', regex=False),
            ua.str.contains('headlesschrome', regex=False),
            ua.str.contains('yabrowser', regex=False),
            ua.str.contains(r'edg/|edga/|edgios/', regex=True),
            ua.str.contains(r'opr/|opera', regex=True),
            ua.str.contains(r'firefox|fxios', regex=True),
            ua.str.contains(r'chrome|crios', regex=True),
            ua.str.contains('safari', regex=False)
        ],
        ['avito_app', 'headless_chrome', 'yandex','edge', 'opera', 'firefox', 'chrome', 'safari'],default='other')

    os_family = np.select([ua.str.contains('android', regex=False),
        ua.str.contains(r'iphone|ipad|ipod', regex=True),ua.str.contains('windows', regex=False),
        ua.str.contains(r'macintosh|mac os x', regex=True),ua.str.contains('linux', regex=False)],
        ['android', 'ios', 'windows', 'macos', 'linux'],default='other')

    platform = (ev['platform'].fillna('unknown').str.strip().str.lower()
        .replace({'desktop': 'web', 'iphone': 'ios'}))
    platform = platform.where(platform.isin(['web', 'android', 'ios']), 'other' )

    data = pd.DataFrame({'cookie_id': ev['cookie_id'].to_numpy(),'client': client,
        'os': os_family,'platform': platform.to_numpy(),'user_agent': ev['user_agent'].to_numpy()})

    features = meta[['cookie_id']].set_index('cookie_id').copy()
    categories = {'client': ['avito_app', 'headless_chrome', 'yandex', 'edge',
            'opera', 'firefox', 'chrome', 'safari', 'other'],
        'os': ['android', 'ios', 'windows', 'macos', 'linux', 'other'],
        'platform': ['web', 'android', 'ios', 'other']}

    for field, values in categories.items():
        counts = pd.crosstab(data['cookie_id'], data[field]).reindex(
            index=features.index,columns=values,fill_value=0)
        shares = counts.div(counts.sum(axis=1).replace(0, np.nan),axis=0)
        features = features.join(shares.add_prefix(f'{field}_share_'))

    features['ua_nunique'] = (data.groupby('cookie_id')['user_agent'].nunique().reindex(features.index, fill_value=0))
    return features

In [39]:
X_train_client = X_train.join(build_client_features(train, ev_tr))
X_test_client = X_test.join(build_client_features(test, ev_te))

assert X_train_client.columns.equals(X_test_client.columns)

print('Признаков до:', X_train.shape[1])
print('Признаков после:', X_train_client.shape[1])

Признаков до: 36
Признаков после: 56


In [40]:
base_scores = {}

for period, (fit_mask, valid_mask) in splits.items():
    baseline_model = CatBoostClassifier(
        iterations=600,
        depth=7,
        learning_rate=0.05,
        l2_leaf_reg=3,
        loss_function='Logloss',
        random_seed=0,
        thread_count=1,
        verbose=False,
        allow_writing_files=False
    )

    baseline_model.fit(
        X_train_client.loc[fit_mask],
        y[fit_mask]
    )

    predictions = baseline_model.predict_proba(
        X_train_client.loc[valid_mask]
    )[:, 1]

    base_scores[period] = precision_at_recall(
        y[valid_mask], predictions
    )

In [41]:
client_results = []

for period, (fit_mask, valid_mask) in splits.items():
    candidate = CatBoostClassifier(
        iterations=600,
        depth=7,
        learning_rate=0.05,
        l2_leaf_reg=5,
        loss_function='Logloss',
        random_seed=0,
        thread_count=1,
        verbose=False,
        allow_writing_files=False
    )

    candidate.fit(X_train_client.loc[fit_mask],y[fit_mask])

    predictions = candidate.predict_proba(X_train_client.loc[valid_mask])[:, 1]

    score = precision_at_recall(y[valid_mask], predictions)

    # лучший результат прежней модели из таблицы подбора
    previous_score = search_results.loc[(search_results['depth'] == 7)
        & (search_results['l2_leaf_reg'] == 5),period].iloc[0]

    client_results.append({
        'Период': period,'Без признаков клиента': previous_score,
        'С признаками клиента': score,'Изменение': score - previous_score})

display(pd.DataFrame(client_results).set_index('Период').round(4))

,Без признаков клиента,С признаками клиента,Изменение
Период,,,
14–16 апреля,0.4875,0.4692,-0.0184
17–19 апреля,0.5744,0.6154,0.0410


Добавление признаков клиента повысило P@R≥0.7 на позднем периоде с 0,5744 до 0,6154, но снизило на раннем с 0,4875 до 0,4692.
Среднее по двум периодам выросло с 0,5310 до 0,5423. Выберем новый варианто так как среднее выше.

Снова посмотрим на разные гиперпараметры 

In [42]:
client_search_results, best_client_params = search_catboost(
    X_train_client,y,splits,
    param_grid={'depth': [6, 7, 8],'iterations': [400, 600],
        'l2_leaf_reg': [5, 10]})

display(client_search_results.round(4))

[1/12] {'depth': 6, 'iterations': 400, 'l2_leaf_reg': 5}: среднее=0.5024, минимум=0.4363
[2/12] {'depth': 6, 'iterations': 400, 'l2_leaf_reg': 10}: среднее=0.4853, минимум=0.4322
[3/12] {'depth': 6, 'iterations': 600, 'l2_leaf_reg': 5}: среднее=0.5223, минимум=0.4582
[4/12] {'depth': 6, 'iterations': 600, 'l2_leaf_reg': 10}: среднее=0.4984, минимум=0.4164
[5/12] {'depth': 7, 'iterations': 400, 'l2_leaf_reg': 5}: среднее=0.4977, минимум=0.4409
[6/12] {'depth': 7, 'iterations': 400, 'l2_leaf_reg': 10}: среднее=0.5027, минимум=0.4190
[7/12] {'depth': 7, 'iterations': 600, 'l2_leaf_reg': 5}: среднее=0.5423, минимум=0.4692
[8/12] {'depth': 7, 'iterations': 600, 'l2_leaf_reg': 10}: среднее=0.5304, минимум=0.4521
[9/12] {'depth': 8, 'iterations': 400, 'l2_leaf_reg': 5}: среднее=0.4873, минимум=0.4228
[10/12] {'depth': 8, 'iterations': 400, 'l2_leaf_reg': 10}: среднее=0.4982, минимум=0.4419
[11/12] {'depth': 8, 'iterations': 600, 'l2_leaf_reg': 5}: среднее=0.5106, минимум=0.4190
[12/12] {'dept

,depth,iterations,learning_rate,l2_leaf_reg,14–16 апреля,17–19 апреля,mean_score,min_score
0,7,600,0.05,5,0.4692,0.6154,0.5423,0.4692
1,7,600,0.05,10,0.4521,0.6087,0.5304,0.4521
2,6,600,0.05,5,0.4582,0.5864,0.5223,0.4582
3,8,600,0.05,10,0.4367,0.5926,0.5147,0.4367
4,8,600,0.05,5,0.4190,0.6022,0.5106,0.4190
5,7,400,0.05,10,0.4190,0.5864,0.5027,0.4190
6,6,400,0.05,5,0.4363,0.5685,0.5024,0.4363
7,6,600,0.05,10,0.4164,0.5803,0.4984,0.4164
8,8,400,0.05,10,0.4419,0.5545,0.4982,0.4419
9,7,400,0.05,5,0.4409,0.5545,0.4977,0.4409


### Подборем скорости обучения, числа итераций и L2
так же l2 лучшее было при 5, но посмотри что происходит с <5

In [43]:
learning_grid = [
    {'learning_rate': [0.03],'iterations': [1000],'l2_leaf_reg': [3, 4, 5]},
    {'learning_rate': [0.05],'iterations': [600],'l2_leaf_reg': [3, 4, 5]},
    {'learning_rate': [0.08],'iterations': [600],'l2_leaf_reg': [3, 4, 5]}]

learning_search_results, final_params = search_catboost(
    X_train_client,y,splits,param_grid=learning_grid,fixed_params={'depth': 7})

display(learning_search_results.round(4))

[1/9] {'iterations': 1000, 'l2_leaf_reg': 3, 'learning_rate': 0.03}: среднее=0.4946, минимум=0.4006
[2/9] {'iterations': 1000, 'l2_leaf_reg': 4, 'learning_rate': 0.03}: среднее=0.5147, минимум=0.4220
[3/9] {'iterations': 1000, 'l2_leaf_reg': 5, 'learning_rate': 0.03}: среднее=0.5445, минимум=0.4633
[4/9] {'iterations': 600, 'l2_leaf_reg': 3, 'learning_rate': 0.05}: среднее=0.5456, минимум=0.4628
[5/9] {'iterations': 600, 'l2_leaf_reg': 4, 'learning_rate': 0.05}: среднее=0.5295, минимум=0.4600
[6/9] {'iterations': 600, 'l2_leaf_reg': 5, 'learning_rate': 0.05}: среднее=0.5423, минимум=0.4692
[7/9] {'iterations': 600, 'l2_leaf_reg': 3, 'learning_rate': 0.08}: среднее=0.5063, минимум=0.4127
[8/9] {'iterations': 600, 'l2_leaf_reg': 4, 'learning_rate': 0.08}: среднее=0.4993, минимум=0.4029
[9/9] {'iterations': 600, 'l2_leaf_reg': 5, 'learning_rate': 0.08}: среднее=0.5225, минимум=0.4152


,depth,iterations,learning_rate,l2_leaf_reg,14–16 апреля,17–19 апреля,mean_score,min_score
0,7,600,0.05,3,0.4628,0.6284,0.5456,0.4628
1,7,1000,0.03,5,0.4633,0.6257,0.5445,0.4633
2,7,600,0.05,5,0.4692,0.6154,0.5423,0.4692
3,7,600,0.05,4,0.4600,0.5989,0.5295,0.4600
4,7,600,0.08,5,0.4152,0.6298,0.5225,0.4152
5,7,1000,0.03,4,0.4220,0.6073,0.5147,0.4220
6,7,600,0.08,3,0.4127,0.6000,0.5063,0.4127
7,7,600,0.08,4,0.4029,0.5957,0.4993,0.4029
8,7,1000,0.03,3,0.4006,0.5885,0.4946,0.4006


#### Сравнение по P@R≥0.7

In [44]:
best_from_each = pd.concat([client_search_results.head(1).assign(search='Глубина, итерации и L2'),
    learning_search_results.head(1).assign( search='Скорость обучения, итерации и L2')], ignore_index=True)

best_from_each = best_from_each.sort_values(['mean_score', 'min_score'],ascending=False)

display(best_from_each.set_index('search').round(4))

,depth,iterations,learning_rate,l2_leaf_reg,14–16 апреля,17–19 апреля,mean_score,min_score
search,,,,,,,,
"Скорость обучения, итерации и L2",7,600,0.05,3,0.4628,0.6284,0.5456,0.4628
"Глубина, итерации и L2",7,600,0.05,5,0.4692,0.6154,0.5423,0.4692


тут получли iterations=600,depth=7,learning_rate=0.05,l2_leaf_reg=3,random_seed=0 лучший по средниму, пэтому выбираем его как основную модель

## обучение и Сабмит 56 признаков

In [45]:
def build_all_features(meta, ev):
    behavior = build_features(meta, ev)
    client = build_client_features(meta, ev)

    return behavior.join(client, validate='one_to_one')

X_train_client = build_all_features(train, ev_tr)
X_test_client = build_all_features(test, ev_te)

In [46]:
# Обучая модель на всём train
final_model = CatBoostClassifier(**final_params)
final_model.fit(X_train_client, train['target'].to_numpy())

CatBoostClassifier(allow_writing_files=False, depth=7, iterations=600, l2_leaf_reg=3.0, learning_rate=0.05, loss_function='Logloss', random_seed=0, thread_count=1, verbose=False)

In [47]:
submission = pd.DataFrame({
    'cookie_id': test['cookie_id'].to_numpy(),
    'score': final_model.predict_proba(X_test_client)[:, 1]
})
submission.to_csv('submission_baseline_56.csv', index=False)

print('Сохранён submission_baseline_56.csv, строк:', len(submission))
display(submission.head())

Сохранён submission_baseline_56.csv, строк: 4909


,cookie_id,score
0,ck_315fb710a0e371e7,0.000898
1,ck_a76ee3b3e3e522fd,0.049501
2,ck_94c9a4d382689e82,0.028073
3,ck_8eaf9509ad9462a0,0.001981
4,ck_9a88a5a989cb5bc6,0.014538


скрытый тест получился 0,51347 поробуем его улучшить 

In [48]:
from sklearn.metrics import log_loss, roc_auc_score
from metric import precision_at_recall

valid_mask = train['window_start_ts'].ge('2026-04-17').to_numpy()
fit_mask = ~valid_mask
labels = train['target'].to_numpy()

diagnostic_model = CatBoostClassifier(
    iterations=600,
    depth=7,
    learning_rate=0.05,
    l2_leaf_reg=3,
    loss_function='Logloss',
    random_seed=0,
    thread_count=1,
    verbose=False,
    allow_writing_files=False
)

diagnostic_model.fit(
    X_train_client.loc[fit_mask],
    labels[fit_mask]
)

diagnostic_results = []

for name, mask in [
    ('Обучение: 6–16 апреля', fit_mask),
    ('Валидация: 17–19 апреля', valid_mask)
]:
    predictions = diagnostic_model.predict_proba(
        X_train_client.loc[mask]
    )[:, 1]

    diagnostic_results.append({
        'Выборка': name,
        'Число куки': int(mask.sum()),
        'Доля ботов': labels[mask].mean(),
        'P@R≥0.7': precision_at_recall(labels[mask], predictions),
        'ROC-AUC': roc_auc_score(labels[mask], predictions),
        'Logloss': log_loss(labels[mask], predictions)
    })

display(
    pd.DataFrame(diagnostic_results)
    .set_index('Выборка')
    .round(4)
)

,Число куки,Доля ботов,P@R≥0.7,ROC-AUC,Logloss
Выборка,,,,,
Обучение: 6–16 апреля,9140,0.0809,1.0000,0.9987,0.0475
Валидация: 17–19 апреля,1951,0.0820,0.6284,0.9016,0.1509


Наблюдается выраженный разрыв между обучением и валидацией: P@R≥0.7 снижается с 1,0000 до 0,6284, а Logloss растёт. Это указывает на переобучение
### добавим признаки

In [49]:
def build_session_features(meta, ev):
    data = ev.sort_values(['cookie_id', 'event_ts']).copy()

    data['gap'] = (
        data.groupby('cookie_id')['event_ts']
        .diff().dt.total_seconds()
    )

    data['new_session'] = data['gap'].isna() | data['gap'].gt(1800)
    data['session_id'] = (
        data.groupby('cookie_id')['new_session'].cumsum()
    )

    sessions = (
        data.groupby(['cookie_id', 'session_id'])
        .agg(
            n_events=('event_ts', 'size'),
            start=('event_ts', 'min'),
            end=('event_ts', 'max')
        )
    )

    sessions['duration'] = (
        sessions['end'] - sessions['start']
    ).dt.total_seconds()

    features = (
        sessions.groupby(level='cookie_id')
        .agg(
            session_count=('n_events', 'size'),
            session_events_median=('n_events', 'median'),
            session_events_max=('n_events', 'max'),
            session_duration_median=('duration', 'median'),
            session_duration_max=('duration', 'max')
        )
    )

    total_events = data.groupby('cookie_id').size()
    features['largest_session_share'] = (
        features['session_events_max'] / total_events
    )

    # Первые события сессий не имеют внутрисессионного интервала
    within = data.loc[~data['new_session']]

    within_stats = within.groupby('cookie_id')['gap'].agg(
        within_gap_median='median',
        within_gap_mean='mean',
        within_gap_std='std'
    )

    features = features.join(within_stats)

    return features.reindex(pd.Index(meta['cookie_id']))

In [50]:
X_train_sessions = X_train_client.join(
    build_session_features(train, ev_tr),
    validate='one_to_one'
)

print('Количество признаков:', X_train_sessions.shape[1])

Количество признаков: 65


## обучение и Сабмит 65 признаков

In [51]:
session_results = []

for period, (fit_mask, valid_mask) in splits.items():
    candidate = CatBoostClassifier(
        iterations=600,
        depth=7,
        learning_rate=0.05,
        l2_leaf_reg=3,
        loss_function='Logloss',
        random_seed=0,
        thread_count=1,
        verbose=False,
        allow_writing_files=False
    )

    candidate.fit(
        X_train_sessions.loc[fit_mask],
        y[fit_mask]
    )

    predictions = candidate.predict_proba(
        X_train_sessions.loc[valid_mask]
    )[:, 1]

    score = precision_at_recall(y[valid_mask], predictions)

    previous = base_scores[period]

    session_results.append({
        'Период': period,
        'Без сессий': previous,
        'С сессиями': score,
        'Изменение': score - previous
    })

display(
    pd.DataFrame(session_results)
    .set_index('Период')
    .round(4)
)

,Без сессий,С сессиями,Изменение
Период,,,
14–16 апреля,0.4628,0.4757,0.0129
17–19 апреля,0.6284,0.5707,-0.0577


Добавление признаков сессий снизило средний P@R≥0.7 с 0,5456 до 0,5232: небольшой прирост на раннем периоде сопровождался ухудшением на позднем.

### напишем общую функцию для построение ансамблей 

In [52]:
def predict_catboost_ensemble(
    X_fit,
    y_fit,
    X_predict,
    seeds=(0, 42, 2026),
    model_params=None
):
    """Обучает CatBoost с разными seed и усредняет вероятности класса 1."""
    if not seeds:
        raise ValueError('Нужно указать хотя бы один seed')

    if not X_fit.columns.equals(X_predict.columns):
        raise ValueError('Колонки обучения и предсказания не совпадают')

    params = {
        'iterations': 600,
        'depth': 7,
        'learning_rate': 0.03,
        'l2_leaf_reg': 3,
        'loss_function': 'Logloss',
        'thread_count': 1,
        'verbose': False,
        'allow_writing_files': False
    }
    params.update(model_params or {})

    models = []
    predictions = []

    for seed in seeds:
        model = CatBoostClassifier(
            **{**params, 'random_seed': seed}
        )
        model.fit(X_fit, np.asarray(y_fit))

        predictions.append(
            model.predict_proba(X_predict)[:, 1]
        )
        models.append(model)

        print(f'Модель с seed={seed} обучена', flush=True)

    scores = np.mean(np.stack(predictions), axis=0)

    return scores, models

In [53]:
results = []

for period, (fit_mask, valid_mask) in splits.items():
    scores, _ = predict_catboost_ensemble(
        X_fit=X_train_client.loc[fit_mask],
        y_fit=y[fit_mask],
        X_predict=X_train_client.loc[valid_mask],
        model_params={
            'iterations': 600,
            'depth': 7,
            'learning_rate': 0.05,
            'l2_leaf_reg': 3
        }
    )

    results.append({
        'Период': period,
        'P@R≥0.7': precision_at_recall(y[valid_mask], scores)
    })

display(pd.DataFrame(results).set_index('Период').round(4))

Модель с seed=0 обучена
Модель с seed=42 обучена
Модель с seed=2026 обучена
Модель с seed=0 обучена
Модель с seed=42 обучена
Модель с seed=2026 обучена


,P@R≥0.7
Период,
14–16 апреля,0.4377
17–19 апреля,0.6667


Усреднение трёх CatBoost повысило средний P@R≥0.7
с 0,5456 до 0,5522. Улучшение получено на позднем периоде,
тогда как на раннем качество снизилось.

Ансамбль выбираем как следующий кандидат по среднему результату,
но преимущество небольшое и не подтверждено на обоих периодах.

In [54]:
scores, _ = predict_catboost_ensemble(
    X_fit=X_train_client,
    y_fit=train['target'].to_numpy(),
    X_predict=X_test_client,
    seeds=(0, 42, 2026),
    model_params={
        'iterations': 600,
        'depth': 7,
        'learning_rate': 0.05,
        'l2_leaf_reg': 3
    }
)

submission_ensemble = pd.DataFrame({'cookie_id': X_test_client.index.to_numpy(),'score': scores})

submission_ensemble.to_csv('submission_ensemble.csv', index=False)

print('Сохранён submission_ensemble.csv, строк:', len(submission_ensemble))
display(submission_ensemble.head())

Модель с seed=0 обучена
Модель с seed=42 обучена
Модель с seed=2026 обучена
Сохранён submission_ensemble.csv, строк: 4909


,cookie_id,score
0,ck_315fb710a0e371e7,0.001934
1,ck_a76ee3b3e3e522fd,0.045871
2,ck_94c9a4d382689e82,0.022604
3,ck_8eaf9509ad9462a0,0.002154
4,ck_9a88a5a989cb5bc6,0.013605


Точность чуть повыселась, попробую еще

### Признаки координат курсора
Добавим доступность координат, разнообразие и повторяемость точек,а также разброс по осям. Так как у бота они будутотличаться от пользователя, скорее всего

In [55]:
def build_pointer_features(meta, ev):
    features = meta[['cookie_id']].set_index('cookie_id').copy()

    # Только события, у которых заполнены обе координаты
    points = ev.loc[
        ev['pointer_x'].notna() & ev['pointer_y'].notna(),
        ['cookie_id', 'pointer_x', 'pointer_y']
    ].copy()

    total_events = (
        ev.groupby('cookie_id').size()
        .reindex(features.index, fill_value=0)
    )

    n_points = (
        points.groupby('cookie_id').size()
        .reindex(features.index, fill_value=0)
    )

    unique_points = (
        points.drop_duplicates(['cookie_id', 'pointer_x', 'pointer_y'])
        .groupby('cookie_id').size()
        .reindex(features.index, fill_value=0)
    )

    features['pointer_n_events'] = n_points
    features['pointer_event_share'] = (
        n_points / total_events.replace(0, np.nan)
    )
    features['pointer_n_unique'] = unique_points

    # Не определена для куки без координат
    features['pointer_repeat_share'] = (
        1 - unique_points / n_points.replace(0, np.nan)
    )

    spread = points.groupby('cookie_id').agg(
        pointer_x_std=('pointer_x', 'std'),
        pointer_y_std=('pointer_y', 'std')
    )
    features = features.join(spread)

    max_point_count = (
        points.groupby(['cookie_id', 'pointer_x', 'pointer_y'])
        .size()
        .groupby(level='cookie_id')
        .max()
    )

    features['pointer_top_point_share'] = (
        max_point_count.reindex(features.index)
        / n_points.replace(0, np.nan)
    )

    return features

In [56]:
X_train_pointer = X_train_client.join(
    build_pointer_features(train, ev_tr),
    validate='one_to_one'
)

assert X_train_pointer.index.equals(X_train_client.index)
assert not np.isinf(X_train_pointer.to_numpy()).any()

print('Признаков до:', X_train_client.shape[1])
print('Признаков после:', X_train_pointer.shape[1])

Признаков до: 56
Признаков после: 63


## обучение и Сабмит 63 признаков
убрали предыдущие и добавили признаки по   координатам

In [57]:
pointer_rows = []

for period, (fit_mask, valid_mask) in splits.items():
    candidate = CatBoostClassifier(
        iterations=600,
        depth=7,
        learning_rate=0.03,
        l2_leaf_reg=3,
        loss_function='Logloss',
        random_seed=0,
        thread_count=1,
        verbose=False,
        allow_writing_files=False
    )

    candidate.fit(
        X_train_pointer.loc[fit_mask],
        y[fit_mask]
    )

    predictions = candidate.predict_proba(
        X_train_pointer.loc[valid_mask]
    )[:, 1]

    score = precision_at_recall(y[valid_mask], predictions)

    previous_score = base_scores[period]

    pointer_rows.append({
        'Период': period,
        'Без координат': previous_score,
        'С координатами': score,
        'Изменение': score - previous_score
    })

pointer_results = pd.DataFrame(pointer_rows).set_index('Период')

display(pointer_results.round(4))
print(
    'Средняя метрика с координатами:',
    round(pointer_results['С координатами'].mean(), 4)
)

,Без координат,С координатами,Изменение
Период,,,
14–16 апреля,0.4628,0.6284,0.1656
17–19 апреля,0.6284,0.7635,0.1351


Средняя метрика с координатами: 0.696


Добавление координат и снижение learning_rate с 0,05 до 0,03 повысили среднюю метрику с 0,5456 до 0,6960.

In [58]:
seeds = [0, 42, 2026]
pointer_seed_rows = []
pointer_ensemble_rows = []

assert X_train_pointer.shape[1] == 63

for period, (fit_mask, valid_mask) in splits.items():
    X_valid = X_train_pointer.loc[valid_mask]

    ensemble_predictions, models = predict_catboost_ensemble(
        X_fit=X_train_pointer.loc[fit_mask],y_fit=y[fit_mask],X_predict=X_valid,
        seeds=seeds,model_params={
            'iterations': 600,
            'depth': 7,
            'learning_rate': 0.03,
            'l2_leaf_reg': 3
        })

    single_scores = []

    for seed, model in zip(seeds, models):
        predictions = model.predict_proba(X_valid)[:, 1]
        score = precision_at_recall(y[valid_mask], predictions)
        single_scores.append(score)

        pointer_seed_rows.append({'Период': period,'Seed': seed,'P@R≥0.7': score})

    single_score = single_scores[0]  
    ensemble_score = precision_at_recall(y[valid_mask], ensemble_predictions)

    pointer_ensemble_rows.append({
        'Период': period,'Одна модель, 63 признака': single_score,
        'Ансамбль, 63 признака': ensemble_score,'Изменение': ensemble_score - single_score})

pointer_seed_results = pd.DataFrame(pointer_seed_rows)
pointer_ensemble_results = (
    pd.DataFrame(pointer_ensemble_rows).set_index('Период')
)

display(pointer_seed_results.pivot(index='Период',columns='Seed',
        values='P@R≥0.7').round(4))

display(pointer_ensemble_results.round(4))

print('Среднее одной модели:',round(pointer_ensemble_results['Одна модель, 63 признака'].mean(), 4))
print('Среднее ансамбля:',round(pointer_ensemble_results['Ансамбль, 63 признака'].mean(), 4))

Модель с seed=0 обучена
Модель с seed=42 обучена
Модель с seed=2026 обучена
Модель с seed=0 обучена
Модель с seed=42 обучена
Модель с seed=2026 обучена


Seed,0,42,2026
Период,,,
14–16 апреля,0.6284,0.6618,0.6035
17–19 апреля,0.7635,0.7778,0.8000


,"Одна модель, 63 признака","Ансамбль, 63 признака",Изменение
Период,,,
14–16 апреля,0.6284,0.6372,0.0088
17–19 апреля,0.7635,0.8058,0.0422


Среднее одной модели: 0.696
Среднее ансамбля: 0.7215


На 63 признаках ансамбль с seed 0, 42 и 2026 улучшил P@R≥0.7 на обоих временных периодах. Средняя метрика выросла с 0,6960 до 0,7215. Выбираем ансамбль следующим кандидатом для отправки

In [59]:
X_train_final = build_all_features(train, ev_tr).join(build_pointer_features(train, ev_tr),validate='one_to_one')

X_test_final = build_all_features(test, ev_te).join(build_pointer_features(test, ev_te),validate='one_to_one')


# Проверяем, что воспроизвели признаки последнего эксперимента
pd.testing.assert_frame_equal(X_train_final, X_train_pointer)

seeds = [0, 42, 2026]
test_predictions = []

for seed in seeds:
    model = CatBoostClassifier(
        iterations=600,
        depth=7,
        learning_rate=0.03,
        l2_leaf_reg=3,
        loss_function='Logloss',
        random_seed=seed,
        thread_count=1,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(X_train_final,train['target'].to_numpy())

    test_predictions.append(model.predict_proba(X_test_final)[:, 1])

    print(f'Модель с seed={seed} обучена', flush=True)

submission_final = pd.DataFrame({'cookie_id': test['cookie_id'].to_numpy(),
                                 'score': np.mean(np.stack(test_predictions), axis=0)})


output_file = 'submission.csv'
submission_final.to_csv(output_file, index=False)

print(f'Сохранён {output_file}, строк: {len(submission_final)}')
display(submission_final.head())

Модель с seed=0 обучена
Модель с seed=42 обучена
Модель с seed=2026 обучена
Сохранён submission.csv, строк: 4909


,cookie_id,score
0,ck_315fb710a0e371e7,0.000773
1,ck_a76ee3b3e3e522fd,0.087216
2,ck_94c9a4d382689e82,0.031166
3,ck_8eaf9509ad9462a0,0.000889
4,ck_9a88a5a989cb5bc6,0.016188


### попробуем добавить кватили

In [60]:
def build_gap_quantiles(meta, ev):
    data = ev.sort_values(['cookie_id', 'event_ts']).copy()

    data['gap_seconds'] = (data.groupby('cookie_id')['event_ts'].diff().dt.total_seconds())

    quantiles = (
        data.groupby('cookie_id')['gap_seconds'].quantile([0.10, 0.25, 0.75, 0.90])
        .unstack().rename(columns={0.10: 'gap_q10',0.25: 'gap_q25',
        0.75: 'gap_q75',0.90: 'gap_q90'}))

    quantiles['gap_iqr'] = (quantiles['gap_q75'] - quantiles['gap_q25'])
    return quantiles.reindex(pd.Index(meta['cookie_id']))


X_train_quantiles = X_train_pointer.join(build_gap_quantiles(train, ev_tr),validate='one_to_one')

In [61]:
quantile_rows = []

for period, (fit_mask, valid_mask) in splits.items():
    candidate = CatBoostClassifier(
        iterations=600,
        depth=7,
        learning_rate=0.03,
        l2_leaf_reg=3,
        loss_function='Logloss',
        random_seed=0,
        thread_count=1,
        verbose=False,
        allow_writing_files=False
    )

    candidate.fit( X_train_quantiles.loc[fit_mask],y[fit_mask])

    predictions = candidate.predict_proba(X_train_quantiles.loc[valid_mask])[:, 1]

    score = precision_at_recall(y[valid_mask], predictions)
    previous_score = pointer_seed_results.loc[
        pointer_seed_results['Период'].eq(period)& pointer_seed_results['Seed'].eq(0),'P@R≥0.7'].iloc[0]

    quantile_rows.append({
        'Период': period,'63 признака': previous_score,'68 признаков': score,
        'Изменение': score - previous_score})

quantile_results = pd.DataFrame(quantile_rows).set_index('Период')
display(quantile_results.round(4))

print('Средняя метрика с квантилями:',round(quantile_results['68 признаков'].mean(), 4))

,63 признака,68 признаков,Изменение
Период,,,
14–16 апреля,0.6284,0.6432,0.0148
17–19 апреля,0.7635,0.7368,-0.0267


Средняя метрика с квантилями: 0.69


точность упала занчит не будем добавлять к признакам и оставим прошлый варинат

## Итоговые результаты

Основная метрика — Precision при Recall ≥ 70%.

| Этап решения | Число признаков | 14–16 апреля | 17–19 апреля | Среднее |
|---|---:|---:|---:|---:|
| Baseline: Random Forest после очистки | 2 | 0,0996 | 0,0957 | 0,0977 |
| Random Forest | 36 | 0,3658 | 0,3972 | 0,3815 |
| HistGradientBoosting | 36 | 0,4164 | 0,4828 | 0,4496 |
| CatBoost после подбора | 36 | 0,4875 | 0,5744 | 0,5310 |
| CatBoost с признаками клиента, LR=0,05 | 56 | 0,4628 | 0,6284 | 0,5456 |
| Ансамбль трёх CatBoost, LR=0,05 | 56 | 0,4377 | 0,6667 | 0,5522 |
| CatBoost с координатами, LR=0,03 | 63 | 0,6284 | 0,7635 | 0,6960 |
| **Итоговый ансамбль, LR=0,03** | **63** | **0,6372** | **0,8058** | **0,7215** |

Таблица отражает этапы решения, а не сравнение алгоритмов в одинаковых условиях: менялись признаки и параметры моделей. Средние рассчитаны до округления.

### Выбранное решение

Итоговая модель - ансамбль трёх CatBoost с seed 0, 42 и 2026. Параметры каждой модели: iterations=600, depth=7, learning_rate=0,03, l2_leaf_reg=3. Предсказания усредняются с равными весами.

Используются 63 признака: возраст куки, количество и состав действий, интервалы, разнообразие объектов, глубина поиска, признаки клиентаи характеристики координат курсора. Проверенные признаки сессийи квантили интервалов не улучшили среднюю метрику и исключены.

Для финального предсказания все три модели обучены на всём train. Результат ансамбля на скрытом тесте - P@R≥0.7 = 0,73077.
